In [7]:
# ============================================================
# LIMPIEZA DE DATOS - TESIS
# Fuente:
# /content/drive/MyDrive/Colab_Notebooks/Tesis/Version_3/dataset_original.xlsx
#
# Salida:
# /content/drive/MyDrive/Colab_Notebooks/Tesis/Version_3/dataset_limpio.xlsx
# ============================================================

# ------------------------------------------------------------
# 1. Montar Google Drive
# ------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# 2. Importar librerías
# ------------------------------------------------------------
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# 3. Definir rutas
# ------------------------------------------------------------
INPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/Tesis/Version_3/dataset_original.xlsx"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/Tesis/Version_3/dataset_limpio.xlsx"

# ------------------------------------------------------------
# 4. Leer archivo
# ------------------------------------------------------------
df = pd.read_excel(INPUT_PATH)

print("Registros originales:", len(df))
print("Columnas originales:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 5. Normalizar nombres de columnas
# ------------------------------------------------------------
def normalize_col(col):
    col = str(col).strip()
    reemplazos = {
        'á':'a','é':'e','í':'i','ó':'o','ú':'u',
        'Á':'A','É':'E','Í':'I','Ó':'O','Ú':'U',
        'ñ':'n','Ñ':'N'
    }
    for a, b in reemplazos.items():
        col = col.replace(a, b)
    col = re.sub(r'[^A-Za-z0-9]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    return col.lower()

df.columns = [normalize_col(c) for c in df.columns]

print("\nColumnas normalizadas:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 6. Homologar columnas esperadas
# ------------------------------------------------------------
aliases = {
    "fecha": ["fecha"],
    "diasemana": ["diasemana", "dia_semana", "dia_de_la_semana"],
    "tipo_plato": ["tipo_plato", "tipoplato", "tipo_de_plato"],
    "clientes": ["clientes", "cantidad_clientes", "cantidad", "platos_vendidos"],
    "facturacion": ["facturacion", "facturacion_diaria", "venta", "ventas", "ingreso_diario"],
    "preciomenu": ["preciomenu", "precio_menu", "menu_precio"],
    "preciosopa": ["preciosopa", "precio_sopa", "sopa_precio"],
    "fanesca_precio": ["fanesca_precio", "precio_fanesca"],
    "coladamorada_precio": ["coladamorada_precio", "precio_coladamorada", "colada_morada_precio", "precio_colada_morada"]
}

resolved = {}
for target, options in aliases.items():
    for opt in options:
        if opt in df.columns:
            resolved[target] = opt
            break

required = ["fecha"]
faltantes = [c for c in required if c not in resolved]
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias: {faltantes}. Columnas detectadas: {df.columns.tolist()}")

for target, source in resolved.items():
    if target != source:
        df[target] = df[source]

for col in ["diasemana", "tipo_plato", "clientes", "facturacion", "preciomenu", "preciosopa", "fanesca_precio", "coladamorada_precio"]:
    if col not in df.columns:
        df[col] = np.nan

df = df[[
    "fecha", "diasemana", "tipo_plato", "clientes", "facturacion",
    "preciomenu", "preciosopa", "fanesca_precio", "coladamorada_precio"
]].copy()

# ------------------------------------------------------------
# 7. Funciones auxiliares
# ------------------------------------------------------------
def to_numeric_clean(series):
    s = series.astype(str).str.strip()
    s = s.replace(["", "nan", "None", "NULL", "NaN"], np.nan)
    s = s.str.replace("$", "", regex=False)
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

def normalize_tipo_plato(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = s.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    s = s.replace(" ", "_")
    mapping = {
        "almuerzo": "almuerzo",
        "menu": "almuerzo",
        "menú": "almuerzo",
        "sopa": "sopa",
        "fanesca": "fanesca",
        "colada_morada": "colada_morada",
        "coladamorada": "colada_morada",
        "colada": "colada_morada",
        "???": np.nan,
        "": np.nan
    }
    return mapping.get(s, s if s in ["almuerzo", "sopa", "fanesca", "colada_morada"] else np.nan)

def normalize_diasemana(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = s.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    mapa = {
        "lun": "Lun", "lunes": "Lun",
        "mar": "Mar", "martes": "Mar",
        "mie": "Mie", "miercoles": "Mie", "miércoles": "Mie",
        "jue": "Jue", "jueves": "Jue",
        "vie": "Vie", "viernes": "Vie",
        "sab": "Sab", "sábado": "Sab", "sabado": "Sab",
        "dom": "Dom", "domingo": "Dom"
    }
    return mapa.get(s, np.nan)

def weekday_3letters(fecha):
    dias = {
        0: "Lun",
        1: "Mar",
        2: "Mie",
        3: "Jue",
        4: "Vie",
        5: "Sab",
        6: "Dom"
    }
    return dias.get(fecha.weekday(), np.nan)

def close_equal(a, b, tol=0.05):
    return pd.notna(a) and pd.notna(b) and abs(a - b) <= tol

def get_precio_menu(fecha):
    anio = fecha.year
    mes = fecha.month
    if anio == 2023:
        return 4.0
    elif anio == 2024:
        return 4.0 if mes <= 8 else 4.5
    elif anio == 2025:
        return 4.5 if mes <= 7 else 5.0
    return np.nan

def get_precio_sopa(fecha):
    anio = fecha.year
    mes = fecha.month
    if anio == 2023:
        return 1.5
    elif anio == 2024:
        return 1.5 if mes <= 8 else 1.8
    elif anio == 2025:
        return 1.8 if mes <= 7 else 2.0
    return np.nan

def get_precio_fanesca(fecha):
    anio = fecha.year
    if anio == 2023:
        return 7.0
    elif anio == 2024:
        return 8.5
    elif anio == 2025:
        return 10.0
    return np.nan

def get_precio_colada(fecha):
    anio = fecha.year
    if anio == 2023:
        return 2.0
    elif anio == 2024:
        return 2.8
    elif anio == 2025:
        return 3.5
    return np.nan

def precio_por_tipo(fecha, tipo_plato, row):
    if tipo_plato == "almuerzo":
        return row["preciomenu"] if pd.notna(row["preciomenu"]) else get_precio_menu(fecha)
    elif tipo_plato == "sopa":
        return row["preciosopa"] if pd.notna(row["preciosopa"]) else get_precio_sopa(fecha)
    elif tipo_plato == "fanesca":
        return row["fanesca_precio"] if pd.notna(row["fanesca_precio"]) else get_precio_fanesca(fecha)
    elif tipo_plato == "colada_morada":
        return row["coladamorada_precio"] if pd.notna(row["coladamorada_precio"]) else get_precio_colada(fecha)
    return np.nan

# ------------------------------------------------------------
# 8. Convertir tipos de datos
# ------------------------------------------------------------
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["clientes"] = to_numeric_clean(df["clientes"])
df["facturacion"] = to_numeric_clean(df["facturacion"])
df["preciomenu"] = to_numeric_clean(df["preciomenu"])
df["preciosopa"] = to_numeric_clean(df["preciosopa"])
df["fanesca_precio"] = to_numeric_clean(df["fanesca_precio"])
df["coladamorada_precio"] = to_numeric_clean(df["coladamorada_precio"])
df["tipo_plato"] = df["tipo_plato"].apply(normalize_tipo_plato)
df["diasemana"] = df["diasemana"].apply(normalize_diasemana)

# ------------------------------------------------------------
# 9. Eliminar registros con fecha nula, inválida o fuera de rango
# ------------------------------------------------------------
antes = len(df)
df = df.dropna(subset=["fecha"]).copy()
elim_fecha_nula = antes - len(df)

antes = len(df)
df = df[(df["fecha"].dt.year >= 2023) & (df["fecha"].dt.year <= 2025)].copy()
elim_fuera_rango = antes - len(df)

print("\nEliminados por fecha nula o inválida:", elim_fecha_nula)
print("Eliminados por fecha fuera de rango:", elim_fuera_rango)

# ------------------------------------------------------------
# 10. Corregir DiaSemana
# ------------------------------------------------------------
validos_laborales = ["Lun", "Mar", "Mie", "Jue", "Vie"]
mask_diasemana = ~df["diasemana"].isin(validos_laborales)
df.loc[mask_diasemana, "diasemana"] = df.loc[mask_diasemana, "fecha"].apply(weekday_3letters)

print("Registros corregidos en DiaSemana:", int(mask_diasemana.sum()))

# ------------------------------------------------------------
# 11. Completar precios vacíos según tipo de plato y fecha
# ------------------------------------------------------------
mask = df["preciomenu"].isna() & (df["tipo_plato"] == "almuerzo")
df.loc[mask, "preciomenu"] = df.loc[mask, "fecha"].apply(get_precio_menu)

mask = df["preciosopa"].isna() & (df["tipo_plato"] == "sopa")
df.loc[mask, "preciosopa"] = df.loc[mask, "fecha"].apply(get_precio_sopa)

mask = df["fanesca_precio"].isna() & (df["tipo_plato"] == "fanesca")
df.loc[mask, "fanesca_precio"] = df.loc[mask, "fecha"].apply(get_precio_fanesca)

mask = df["coladamorada_precio"].isna() & (df["tipo_plato"] == "colada_morada")
df.loc[mask, "coladamorada_precio"] = df.loc[mask, "fecha"].apply(get_precio_colada)

print("Precios completados según reglas de negocio.")

# ------------------------------------------------------------
# 12. Recuperar Tipo Plato si está vacío
# ------------------------------------------------------------
mask_tipo_vacio = df["tipo_plato"].isna()
mask_recuperable = (
    mask_tipo_vacio &
    df["facturacion"].notna() &
    df["clientes"].notna() &
    (df["clientes"] > 0)
)

recuperados_tipo = 0

for idx in df[mask_recuperable].index:
    fecha = df.at[idx, "fecha"]
    facturacion = df.at[idx, "facturacion"]
    clientes = df.at[idx, "clientes"]

    valor_unitario = facturacion / clientes

    precio_menu = df.at[idx, "preciomenu"] if pd.notna(df.at[idx, "preciomenu"]) else get_precio_menu(fecha)
    precio_sopa = df.at[idx, "preciosopa"] if pd.notna(df.at[idx, "preciosopa"]) else get_precio_sopa(fecha)
    precio_fanesca = df.at[idx, "fanesca_precio"] if pd.notna(df.at[idx, "fanesca_precio"]) else get_precio_fanesca(fecha)
    precio_colada = df.at[idx, "coladamorada_precio"] if pd.notna(df.at[idx, "coladamorada_precio"]) else get_precio_colada(fecha)

    if close_equal(valor_unitario, precio_sopa):
        df.at[idx, "tipo_plato"] = "sopa"
        df.at[idx, "preciosopa"] = precio_sopa
        recuperados_tipo += 1
    elif close_equal(valor_unitario, precio_menu):
        df.at[idx, "tipo_plato"] = "almuerzo"
        df.at[idx, "preciomenu"] = precio_menu
        recuperados_tipo += 1
    elif close_equal(valor_unitario, precio_fanesca):
        df.at[idx, "tipo_plato"] = "fanesca"
        df.at[idx, "fanesca_precio"] = precio_fanesca
        recuperados_tipo += 1
    elif close_equal(valor_unitario, precio_colada):
        df.at[idx, "tipo_plato"] = "colada_morada"
        df.at[idx, "coladamorada_precio"] = precio_colada
        recuperados_tipo += 1

print("Registros recuperados en Tipo Plato:", recuperados_tipo)

# ------------------------------------------------------------
# 13. Completar Facturación si está vacía y Clientes tiene valor
# ------------------------------------------------------------
mask_facturacion_vacia = df["facturacion"].isna() & df["clientes"].notna() & (df["clientes"] > 0)

completados_fact = 0

for idx in df[mask_facturacion_vacia].index:
    fecha = df.at[idx, "fecha"]
    tipo = df.at[idx, "tipo_plato"]
    clientes = df.at[idx, "clientes"]

    if pd.isna(tipo):
        continue

    precio = precio_por_tipo(fecha, tipo, df.loc[idx])
    if pd.notna(precio):
        if tipo == "almuerzo":
            df.at[idx, "preciomenu"] = precio
        elif tipo == "sopa":
            df.at[idx, "preciosopa"] = precio
        elif tipo == "fanesca":
            df.at[idx, "fanesca_precio"] = precio
        elif tipo == "colada_morada":
            df.at[idx, "coladamorada_precio"] = precio

        df.at[idx, "facturacion"] = precio * clientes
        completados_fact += 1

print("Registros con facturación completada:", completados_fact)

# ------------------------------------------------------------
# 14. Corregir facturación mayor a 10000
# ------------------------------------------------------------
mask_facturacion_alta = df["facturacion"].notna() & (df["facturacion"] > 10000)

corregidos_facturacion = 0

for idx in df[mask_facturacion_alta].index:
    fecha = df.at[idx, "fecha"]
    tipo = df.at[idx, "tipo_plato"]
    clientes = df.at[idx, "clientes"]

    if pd.isna(tipo) or pd.isna(clientes):
        continue

    precio = precio_por_tipo(fecha, tipo, df.loc[idx])
    if pd.notna(precio):
        if tipo == "almuerzo":
            df.at[idx, "preciomenu"] = precio
        elif tipo == "sopa":
            df.at[idx, "preciosopa"] = precio
        elif tipo == "fanesca":
            df.at[idx, "fanesca_precio"] = precio
        elif tipo == "colada_morada":
            df.at[idx, "coladamorada_precio"] = precio

        df.at[idx, "facturacion"] = clientes * precio
        corregidos_facturacion += 1

print("Registros con facturación > 10000 corregidos:", corregidos_facturacion)

# ------------------------------------------------------------
# 15. Corregir clientes si está vacío o es menor a 0
# ------------------------------------------------------------
mask_clientes_invalidos = (df["clientes"].isna() | (df["clientes"] < 0)) & df["facturacion"].notna()

corregidos_clientes = 0

for idx in df[mask_clientes_invalidos].index:
    fecha = df.at[idx, "fecha"]
    tipo = df.at[idx, "tipo_plato"]
    facturacion = df.at[idx, "facturacion"]

    if pd.isna(tipo) or pd.isna(facturacion):
        continue

    precio = precio_por_tipo(fecha, tipo, df.loc[idx])
    if pd.notna(precio) and precio > 0:
        if tipo == "almuerzo":
            df.at[idx, "preciomenu"] = precio
        elif tipo == "sopa":
            df.at[idx, "preciosopa"] = precio
        elif tipo == "fanesca":
            df.at[idx, "fanesca_precio"] = precio
        elif tipo == "colada_morada":
            df.at[idx, "coladamorada_precio"] = precio

        df.at[idx, "clientes"] = facturacion / precio
        corregidos_clientes += 1

print("Registros con clientes vacíos o negativos corregidos:", corregidos_clientes)

# ------------------------------------------------------------
# 16. Eliminar registros donde clientes y facturación estén vacíos
# ------------------------------------------------------------
antes = len(df)
mask_ambos_vacios = df["clientes"].isna() & df["facturacion"].isna()
df = df[~mask_ambos_vacios].copy()
eliminados_ambos_vacios = antes - len(df)

print("Registros eliminados por clientes y facturación vacíos:", eliminados_ambos_vacios)

# ------------------------------------------------------------
# 17. Eliminar registros extremos
# ------------------------------------------------------------
antes = len(df)
mask_extremos = (df["clientes"] > 300) & (df["facturacion"] > 10000)
df = df[~mask_extremos].copy()
eliminados_extremos = antes - len(df)

print("Registros eliminados por clientes > 300 y facturación > 10000:", eliminados_extremos)

# ------------------------------------------------------------
# 18. Redondear clientes porque representa conteo
# ------------------------------------------------------------
df["clientes"] = df["clientes"].round(0)

# ------------------------------------------------------------
# 19. Eliminar registros cuyo tipo_plato siga vacío
# ------------------------------------------------------------
antes = len(df)
df = df[df["tipo_plato"].notna()].copy()
eliminados_tipo = antes - len(df)

print("Registros eliminados por tipo_plato vacío o irrecuperable:", eliminados_tipo)

# ------------------------------------------------------------
# 20. ELIMINAR DUPLICADOS EXACTOS
# ------------------------------------------------------------
antes = len(df)
df = df.drop_duplicates().copy()
duplicados_exactos = antes - len(df)

print("Duplicados exactos eliminados:", duplicados_exactos)

# ------------------------------------------------------------
# 21. ELIMINAR DUPLICADOS LÓGICOS DE NEGOCIO
#    Caso típico:
#    misma fecha + mismo tipo_plato + mismos clientes + misma facturación
# ------------------------------------------------------------
antes = len(df)
df = df.sort_values(
    by=["fecha", "tipo_plato", "clientes", "facturacion"],
    ascending=[True, True, False, False]
).copy()

subset_dupes = ["fecha", "tipo_plato", "clientes", "facturacion"]
df = df.drop_duplicates(subset=subset_dupes, keep="first").copy()
duplicados_logicos = antes - len(df)

print("Duplicados lógicos eliminados:", duplicados_logicos)

# ------------------------------------------------------------
# 22. ELIMINAR DUPLICADOS POR FECHA + TIPO SI TODO COINCIDE EN LO ESENCIAL
#    Esto ayuda cuando cambia una columna auxiliar pero el registro es el mismo
# ------------------------------------------------------------
antes = len(df)

# redondeo auxiliar para comparar
df["_fact_red"] = df["facturacion"].round(2)
df["_cli_red"] = df["clientes"].round(0)

subset_dupes_2 = ["fecha", "tipo_plato", "_cli_red", "_fact_red"]
df = df.drop_duplicates(subset=subset_dupes_2, keep="first").copy()
duplicados_equivalentes = antes - len(df)

df = df.drop(columns=["_fact_red", "_cli_red"])

print("Duplicados equivalentes eliminados:", duplicados_equivalentes)

# ------------------------------------------------------------
# 23. VALIDACIÓN FINAL DE DUPLICADOS POR FECHA Y TIPO
# ------------------------------------------------------------
revisar_dupes = (
    df.groupby(["fecha", "tipo_plato"])
      .size()
      .reset_index(name="conteo")
      .sort_values("conteo", ascending=False)
)

posibles_conflictos = revisar_dupes[revisar_dupes["conteo"] > 1].copy()

print("\nCantidad de combinaciones fecha + tipo_plato repetidas:", len(posibles_conflictos))

if len(posibles_conflictos) > 0:
    print("\nAdvertencia: aún existen combinaciones repetidas de fecha + tipo_plato.")
    print("Revisa si corresponden a ventas separadas reales o a duplicados no resueltos.")
    print(posibles_conflictos.head(20).to_string(index=False))
else:
    print("No quedan combinaciones repetidas de fecha + tipo_plato.")

# ------------------------------------------------------------
# 24. Ordenar dataset final
# ------------------------------------------------------------
df = df.sort_values(["fecha", "tipo_plato"]).reset_index(drop=True)

print("\nTotal registros finales:", len(df))
print("\nConteo por tipo de plato:")
print(df["tipo_plato"].value_counts(dropna=False))

print("\nVista previa final:")
print(df.head())

# ------------------------------------------------------------
# 25. RESUMEN FINAL DE LIMPIEZA
# ------------------------------------------------------------
print("\n================ RESUMEN DE LIMPIEZA ================")
print("Eliminados por fecha nula o inválida:", elim_fecha_nula)
print("Eliminados por fecha fuera de rango:", elim_fuera_rango)
print("Registros corregidos en DiaSemana:", int(mask_diasemana.sum()))
print("Registros recuperados en Tipo Plato:", recuperados_tipo)
print("Registros con facturación completada:", completados_fact)
print("Registros con facturación > 10000 corregidos:", corregidos_facturacion)
print("Registros con clientes vacíos o negativos corregidos:", corregidos_clientes)
print("Registros eliminados por clientes y facturación vacíos:", eliminados_ambos_vacios)
print("Registros eliminados por extremos:", eliminados_extremos)
print("Registros eliminados por tipo_plato vacío:", eliminados_tipo)
print("Duplicados exactos eliminados:", duplicados_exactos)
print("Duplicados lógicos eliminados:", duplicados_logicos)
print("Duplicados equivalentes eliminados:", duplicados_equivalentes)
print("Total registros finales:", len(df))

# ------------------------------------------------------------
# 26. Guardar archivo limpio
# ------------------------------------------------------------
df.to_excel(OUTPUT_PATH, index=False)
print(f"\nArchivo limpio guardado en:\n{OUTPUT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Registros originales: 1739
Columnas originales:
['Fecha', 'DíaSemana', 'PrecioMenú', 'PrecioSopa', 'Fanesca_Precio', 'ColadaMorada_Precio', 'Clientes', 'Tipo Plato', 'Facturación']

Columnas normalizadas:
['fecha', 'diasemana', 'preciomenu', 'preciosopa', 'fanesca_precio', 'coladamorada_precio', 'clientes', 'tipo_plato', 'facturacion']

Eliminados por fecha nula o inválida: 126
Eliminados por fecha fuera de rango: 0
Registros corregidos en DiaSemana: 79
Precios completados según reglas de negocio.
Registros recuperados en Tipo Plato: 56
Registros con facturación completada: 103
Registros con facturación > 10000 corregidos: 39


/tmp/ipykernel_4771/451960900.py:260: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<DatetimeArray>
[]
Length: 0, dtype: datetime64[ns]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "fanesca_precio"] = df.loc[mask, "fecha"].apply(get_precio_fanesca)
/tmp/ipykernel_4771/451960900.py:263: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<DatetimeArray>
[]
Length: 0, dtype: datetime64[ns]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "coladamorada_precio"] = df.loc[mask, "fecha"].apply(get_precio_colada)


Registros con clientes vacíos o negativos corregidos: 136
Registros eliminados por clientes y facturación vacíos: 11
Registros eliminados por clientes > 300 y facturación > 10000: 5
Registros eliminados por tipo_plato vacío o irrecuperable: 11
Duplicados exactos eliminados: 92
Duplicados lógicos eliminados: 0
Duplicados equivalentes eliminados: 0

Cantidad de combinaciones fecha + tipo_plato repetidas: 5

Advertencia: aún existen combinaciones repetidas de fecha + tipo_plato.
Revisa si corresponden a ventas separadas reales o a duplicados no resueltos.
     fecha tipo_plato  conteo
2025-08-19       sopa       2
2024-10-28   almuerzo       2
2025-05-01       sopa       2
2023-12-13   almuerzo       2
2023-12-13       sopa       2

Total registros finales: 1494

Conteo por tipo de plato:
tipo_plato
almuerzo         739
sopa             664
colada_morada     57
fanesca           34
Name: count, dtype: int64

Vista previa final:
       fecha diasemana tipo_plato  clientes  facturacion  pre